# Movie rating prediction with neural collaborative filtering

This notebook trains a user/item embedding model on MovieLens ratings joined with movie metadata. It requires `ratings.csv`, `links.csv`, and `movies_metadata.csv` in this folder, plus PyTorch, pandas, NumPy, scikit-learn and tqdm. These data files are not included here. Set `MOVIES_DATA_ROOT` to their directory if needed. See [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) for the source and terms. The full original training run took about two hours.

The earlier test score was invalid because the test split was created after training. The revised code splits before training, evaluates only users and items observed during training, and reports ranking metrics over held-out rated items. The results below were generated on the full original dataset using a Kaggle GPU. Repetitive batch progress output was removed; epoch metrics are retained.

**Data:** [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset). Supply `ratings.csv`, `links.csv`, and `movies_metadata.csv` in this folder or set `MOVIES_DATA_ROOT`.

In [1]:
import os
from pathlib import Path
import pandas as pd

data_root = Path(os.environ.get("MOVIES_DATA_ROOT", "."))
ratings = pd.read_csv(data_root / "ratings.csv", dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"})

ratings.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [2]:
metadata = pd.read_csv(data_root / 'movies_metadata.csv', low_memory=False)

metadata.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
links = pd.read_csv(data_root / 'links.csv', dtype={'tmdbId': 'float64'})

links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [4]:
metadata = metadata.dropna(subset=['production_countries', 'imdb_id', 'original_language', 'popularity', 'production_companies', 'production_countries', 'revenue', 'spoken_languages', 'title', 'video', 'vote_average', 'release_date', 'overview'])

metadata['status'] = metadata['status'].fillna(metadata['status'].mode()[0])
metadata['runtime'] = metadata['runtime'].fillna(metadata['runtime'].median())

cols_to_drop = ['belongs_to_collection', 'homepage', 'tagline', 'poster_path']
metadata = metadata.drop(columns=cols_to_drop, errors='ignore')

In [5]:
metadata.isna().sum()

adult                   0
budget                  0
genres                  0
id                      0
imdb_id                 0
original_language       0
original_title          0
overview                0
popularity              0
production_companies    0
production_countries    0
release_date            0
revenue                 0
runtime                 0
spoken_languages        0
status                  0
title                   0
video                   0
vote_average            0
vote_count              0
dtype: int64

In [6]:
ratings = ratings.drop_duplicates(subset=['userId', 'movieId'], keep='last')
ratings = ratings[['userId', 'movieId', 'rating', 'timestamp']].copy()

links = links.dropna(subset=['movieId'])
links['movieId'] = links['movieId'].astype(int)

ratings = ratings[ratings['movieId'].isin(links['movieId'])].copy()

In [7]:
links = pd.read_csv(data_root / 'links.csv', dtype={'tmdbId': 'float64'})
links = links.dropna(subset=['tmdbId'])
links['tmdbId'] = links['tmdbId'].astype(int)
links = links.rename(columns={'tmdbId': 'movie_tmdb_id'})

In [8]:
metadata['id'] = pd.to_numeric(metadata['id'], errors='coerce')
metadata = metadata.dropna(subset=['id'])
metadata['id'] = metadata['id'].astype(int)

metadata = metadata.sort_values('vote_count', ascending=False)
metadata = metadata.drop_duplicates(subset='id', keep='first')

metadata = metadata.rename(columns={'id': 'tmdbId'})
# Only the ID and title are needed for rating prediction.
metadata = metadata[['tmdbId', 'title']]

In [9]:
ratings_with_tmdb = ratings.merge(links[['movieId', 'movie_tmdb_id']],
                                  on='movieId',
                                  how='left')

ratings_with_tmdb = ratings_with_tmdb.dropna(subset=['movie_tmdb_id'])
ratings_with_tmdb['movie_tmdb_id'] = ratings_with_tmdb['movie_tmdb_id'].astype(int)

print(f"Осталось оценок после связи с TMDb: {len(ratings_with_tmdb):,}")

Осталось оценок после связи с TMDb: 26,010,786


In [10]:
ratings_full = ratings_with_tmdb.rename(columns={'movie_tmdb_id': 'tmdbId'})

df = ratings_full.merge(metadata, on='tmdbId', how='left')

print(f"После merge с metadata: {len(df):,}")
print(f"Пропусков в title: {df['title'].isna().sum()}")

После merge с metadata: 26,010,786
Пропусков в title: 34110


In [11]:
df = df.dropna(subset=['title'])
print(f"После дропа title-NaN: {len(df):,}")
# Multiple MovieLens IDs may map to the same TMDb title. Keep the latest rating.
df = df.sort_values("timestamp").drop_duplicates(["userId", "tmdbId"], keep="last").copy()

После дропа title-NaN: 25,976,676


In [12]:
from sklearn.preprocessing import LabelEncoder

if 'user_idx' not in df.columns or 'item_idx' not in df.columns:
    user_le = LabelEncoder()
    item_le = LabelEncoder()

    df['user_idx'] = user_le.fit_transform(df['userId'])
    df['item_idx'] = item_le.fit_transform(df['tmdbId'])

    user_mapping = pd.DataFrame({
        'userId': user_le.classes_,
        'user_idx': range(len(user_le.classes_))
    })
    item_mapping = pd.DataFrame({
        'tmdbId': item_le.classes_,
        'item_idx': range(len(item_le.classes_))
    })

    user_mapping.to_csv('user_mapping.csv', index=False)
    item_mapping.to_csv('item_mapping.csv', index=False)

num_users = df['user_idx'].nunique()
num_items = df['item_idx'].nunique()

print(f"Пользователей: {num_users:,} | Фильмов: {num_items:,}")
print(df[['user_idx', 'item_idx', 'rating']].head(3))

Пользователей: 270,882 | Фильмов: 43,678


         user_idx  item_idx  rating
3660662     38149      1005     4.0
4311765     44716      2711     3.0
4311771     44716       628     5.0


In [13]:
from sklearn.model_selection import train_test_split

# Split before fitting any model. Remove cold-start users/items from evaluation:
# this embedding model cannot make a meaningful prediction for unseen IDs.
data = df[["user_idx", "item_idx", "rating"]].copy()
train_val_df, test_raw = train_test_split(data, test_size=0.15, random_state=42,
                                           stratify=data["rating"].round())
train_df, val_raw = train_test_split(train_val_df, test_size=0.15, random_state=42,
                                     stratify=train_val_df["rating"].round())
known_users = set(train_df["user_idx"].unique())
known_items = set(train_df["item_idx"].unique())
def warm_only(part):
    return part[part["user_idx"].isin(known_users) & part["item_idx"].isin(known_items)].copy()
val_df, test_df = warm_only(val_raw), warm_only(test_raw)
print(f"Train: {len(train_df):,}; validation: {len(val_df):,}; test: {len(test_df):,}")
print(f"Cold-start rows excluded: validation {len(val_raw)-len(val_df):,}; test {len(test_raw)-len(test_df):,}")
assert data.index.is_unique
assert len(train_df) + len(val_raw) + len(test_raw) == len(data)

Train: 18,768,057; validation: 3,309,567; test: 3,893,668
Cold-start rows excluded: validation 2,444; test 2,815


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(42)

class RatingDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items  = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

batch_size = 4096
num_workers = 0

train_ds = RatingDataset(train_df)
val_ds   = RatingDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

In [15]:
# Модель (NCF)
class SimpleNCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, u, i):
        user_vec = self.user_emb(u)
        item_vec = self.item_emb(i)
        x = torch.cat([user_vec, item_vec], dim=-1)
        return self.mlp(x).squeeze(-1)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = SimpleNCF(num_users, num_items, emb_dim=64).to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

Device: cuda


In [16]:
from tqdm import tqdm

# Обучение
epochs = 6
best_val_loss = float('inf')

for epoch in range(epochs):
    start = time.time()

    model.train()
    train_loss = 0.0

    for u, i, r in train_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        pred = model(u, i)
        loss = criterion(pred, r)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(r)


    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for u, i, r in val_loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            pred = model(u, i)
            val_loss += criterion(pred, r).item() * len(r)
    val_loss /= len(val_ds)

    rmse_train = train_loss ** 0.5
    rmse_val   = val_loss ** 0.5

    print(f"[{epoch+1:2d}/{epochs}]  train RMSE: {rmse_train:.4f}  |  val RMSE: {rmse_val:.4f}  |  {time.time()-start:.1f} с")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_ncf_model.pt')
        print("   → сохранена лучшая модель")

[ 1/6]  train RMSE: 0.9343  |  val RMSE: 0.8775  |  309.2 с
   → сохранена лучшая модель
[ 2/6]  train RMSE: 0.8692  |  val RMSE: 0.8862  |  307.9 с
[ 3/6]  train RMSE: 0.8506  |  val RMSE: 0.8513  |  305.1 с
   → сохранена лучшая модель
[ 4/6]  train RMSE: 0.8372  |  val RMSE: 0.8497  |  305.2 с
   → сохранена лучшая модель
[ 5/6]  train RMSE: 0.8279  |  val RMSE: 0.8481  |  304.5 с
   → сохранена лучшая модель
[ 6/6]  train RMSE: 0.8202  |  val RMSE: 0.8414  |  301.8 с
   → сохранена лучшая модель


In [17]:
# The independent test partition was created before training in the split cell.
print(f"Independent test ratings: {len(test_df):,}")

Independent test ratings: 3,893,668


In [18]:
test_ds = RatingDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=4096, shuffle=False, num_workers=0)

In [19]:
model = SimpleNCF(num_users, num_items, emb_dim=64).to(device)

model.load_state_dict(torch.load('best_ncf_model.pt', map_location=device, weights_only=True))
model.eval()

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for u, i, r in test_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        pred = model(u, i)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(r.cpu().numpy())

rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
mae = mean_absolute_error(all_labels, all_preds)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

Test RMSE: 0.8418
Test MAE: 0.6486


The test metrics above measure rating prediction on held-out interactions for users and items seen during training. They do not cover cold-start cases.

In [20]:
from sklearn.metrics import ndcg_score, average_precision_score

def calculate_ranking_metrics(user_idx, top_k=10):
    """Rank only items with observed test ratings; this is not full-catalog ranking."""
    observed = test_df[test_df["user_idx"] == user_idx]
    if len(observed) < 2:
        return None
    relevance = (observed["rating"].to_numpy() >= 4).astype(int)
    if relevance.sum() == 0:
        return None
    items = torch.as_tensor(observed["item_idx"].to_numpy(), dtype=torch.long, device=device)
    users = torch.full_like(items, int(user_idx))
    with torch.no_grad():
        scores = model(users, items).cpu().numpy()
    k = min(top_k, len(observed))
    best = np.argsort(scores)[-k:][::-1]
    precision = relevance[best].mean()
    recall = relevance[best].sum() / relevance.sum()
    return {"ndcg@k": ndcg_score([relevance], [scores], k=k),
            "precision@k": precision, "recall@k": recall,
            "average_precision": average_precision_score(relevance, scores)}

def evaluate_ranking_metrics(n_users=100, top_k=10):
    eligible = test_df.groupby("user_idx").size()
    users = eligible[eligible >= 2].index.to_numpy(copy=True)
    rng = np.random.default_rng(42)
    rng.shuffle(users)
    results = [calculate_ranking_metrics(u, top_k) for u in users[:n_users]]
    results = [r for r in results if r is not None]
    if not results:
        return {}, 0
    return {key: np.mean([r[key] for r in results]) for key in results[0]}, len(results)

ranking_results, evaluated_users = evaluate_ranking_metrics(n_users=100, top_k=10)
print(f"Observed-test-item ranking; evaluated users: {evaluated_users}")
for metric, value in ranking_results.items():
    print(f"{metric}: {value:.4f}")

Observed-test-item ranking; evaluated users: 94
ndcg@k: 0.8498
precision@k: 0.6319
recall@k: 0.8294
average_precision: 0.8169


These ranking scores compare only items with observed test ratings. They are conditional on the test candidates and should not be interpreted as full-catalog recommendation precision.

In [21]:
def calculate_personalization(n_users=50, top_k=10):
    """Mean pairwise Jaccard distance between unseen-item recommendations."""
    rng = np.random.default_rng(42)
    users = np.array(sorted(known_users))
    users = rng.choice(users, size=min(n_users, len(users)), replace=False)
    items = torch.arange(num_items, dtype=torch.long, device=device)
    available = np.isin(np.arange(num_items), list(known_items))
    seen = pd.concat([train_df[train_df["user_idx"].isin(users)], val_df[val_df["user_idx"].isin(users)]])
    seen_by_user = seen.groupby("user_idx")["item_idx"].apply(list)
    recommendations = []
    for user_idx in users:
        with torch.no_grad():
            scores = model(torch.full_like(items, int(user_idx)), items).cpu().numpy()
        scores[~available] = -np.inf
        scores[seen_by_user.get(user_idx, [])] = -np.inf
        candidates = np.flatnonzero(np.isfinite(scores))
        if not len(candidates):
            continue
        top = candidates[np.argsort(scores[candidates])[-top_k:]]
        recommendations.append(set(top))
    similarities = [len(a & b) / len(a | b)
                    for i, a in enumerate(recommendations) for b in recommendations[i + 1:]]
    if not similarities:
        return {"personalization": float("nan"), "average_jaccard_similarity": float("nan")}
    mean_similarity = float(np.mean(similarities))
    return {"personalization": 1 - mean_similarity, "average_jaccard_similarity": mean_similarity}

personalization = calculate_personalization()
for metric, value in personalization.items():
    print(f"{metric}: {value:.4f}")

personalization: 0.6506
average_jaccard_similarity: 0.3494


Personalization is the mean Jaccard distance between recommendation lists for sampled users. A larger value indicates less overlap, not necessarily better recommendations.